# 02 — Train genre classifier (9 parent genres)

Pipeline (matches `scripts/train_model.py --model rf --no-tune`):
1. Clean raw Spotify tracks
2. Drop ambiguous fine-grained tags (mood / vague / entertainment)
3. Collapse remaining labels into **9 parent genres**
4. Drop tracks with conflicting parent labels; keep one row per consistent track
5. Hold out **20%** for testing (stratified)
6. Train a size-friendly Random Forest and save `models/genre_classifier.joblib`

**Prerequisite:** `data/raw/dataset.csv`  
**Canonical CLI:** `python scripts/train_model.py --model rf --no-tune`  
**Defaults:** `n_estimators=100`, `max_depth=20`, `min_samples_leaf=2`, `max_features=0.5`

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path("..").resolve()

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

from src.clean import clean_tracks, print_cleaning_report
from src.data_io import AUDIO_FEATURES, TARGET_COLUMN, load_raw_tracks, raw_csv_exists
from src.genre_map import GENRE_COLLAPSE_MAP, PARENT_GENRES, parent_genre_guide_frame
from src.train import MODEL_PATH, METRICS_PATH, load_metrics, train_genre_classifier

print("Collapse map size:", len(GENRE_COLLAPSE_MAP))
print("Parent genres:", PARENT_GENRES)
display(parent_genre_guide_frame())

In [ ]:
assert raw_csv_exists(), "Missing data/raw/dataset.csv — run download/load first."
df_raw = load_raw_tracks()
print("Raw shape:", df_raw.shape)
df_raw[AUDIO_FEATURES + [TARGET_COLUMN]].head()

## Clean + collapse

Cleaning drops ambiguous fine tags and tracks whose labels disagree across parents after collapse.

In [ ]:
df_clean, report = clean_tracks(
    df_raw,
    min_genre_count=500,
    collapse_genres=True,
    drop_ambiguous_fine_genres=True,
    drop_conflicting_parents=True,
    save=True,
)
print_cleaning_report(report)
df_clean[TARGET_COLUMN].value_counts()

## Train

Uses the same defaults as the committed model / CLI (`--model rf --no-tune`).

In [ ]:
result = train_genre_classifier(
    df=df_raw,
    model="rf",
    tune=False,
    n_estimators=100,
    max_depth=20,
    min_samples_leaf=2,
    min_genre_count=500,
    collapse_genres=True,
)
(
    result["metrics"]["accuracy"],
    result["metrics"]["macro_f1"],
    result["metrics"]["n_classes"],
    result["model_path"],
)

In [ ]:
metrics = load_metrics()
print("Saved model exists:", MODEL_PATH.exists())
print("Accuracy:", round(metrics["accuracy"], 4))
print("Macro F1:", round(metrics["macro_f1"], 4))
print("Weighted F1:", round(metrics["weighted_f1"], 4))
print("Classes:", metrics["n_classes"])
print("Best / used params:", metrics.get("best_params") or {
    "n_estimators": metrics.get("n_estimators"),
    "max_depth": metrics.get("max_depth"),
    "min_samples_leaf": metrics.get("min_samples_leaf"),
})
print("Clean genres used:", metrics.get("cleaning", {}).get("clean_genres"))
print("Genre counts:", metrics.get("cleaning", {}).get("genre_counts"))
print("Metrics file:", METRICS_PATH)